<div dir="rtl" align="right">

# مُرشِّحُ التردداتِ العاليةِ \(High-Pass Filter\)

**مجموعةُ البياناتِ**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**القنواتُ**: P4, Cz, F8, T7  
**معدّلُ أخذِ العيناتِ**: 200 Hz  
**المُشاركُ**: 1 (مُشاركٌ واحدٌ فقط للسرعةِ)

---

## نظرةٌ عامّةٌ

يُزيلُ مُرشِّحُ التردداتِ العاليةِ المكوّناتِ ذاتَ الترددِ المنخفضِ (أقلَّ من 1 Hz) من إشارةِ EEG. ويشملُ ذلك:

- إزاحةَ التيارِ المُستمرِّ (DC offset)
- الانجرافَ الأساسيَّ (baseline drift)
- الشوائبَ ذاتَ الترددِ المنخفضِ (العرقُ، التنفّسُ، حركةُ القطبِ)

## المُخرجاتُ المُتوقّعةُ

بعدَ الترشيحِ، تَستوي الإشارةُ على المحورِ العموديِّ ويختفي الانجرافُ البطيءُ، لكنّها تبقى مُشوّشةً لأنَّ الضجيجَ عاليَ الترددِ لم يُزَلْ بعدُ. سنتعاملُ معهُ في دفترِ مُرشِّحِ التردداتِ المنخفضةِ.

## المُعاملاتُ الرئيسةُ

| المُعاملُ | القيمةُ | الدلالةُ |
| --------- | ------- | ------ |
| ترددُ القطعِ | 1 Hz | تُزالُ التردداتُ أقلَّ من 1 Hz |
| الترتيبُ | 4 | شدّةُ انحدارِ المُرشِّحِ |
| الطريقةُ | Butterworth | استجابةٌ مُسطّحةٌ في نطاقِ التمريرِ |
| filtfilt | نعم | صفريُّ الطورِ (لا تأخيرَ زمنيًّا) |

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install scipy numpy plotly wfdb

<div dir="rtl" align="right">

## 2. استنساخُ المستودعِ وتنزيلُ بياناتِ مُشاركٍ واحدٍ

نُنزِّلُ بياناتِ المُشاركِ 1 فقط لإبقاءِ التنزيلِ سريعًا (~12 تسجيلًا، نحوَ 8 MB). تَحتوي مجموعةُ البياناتِ الكاملةُ على 20 مُشاركًا.

</div>

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')

In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 7

<div dir="rtl" align="right">

## 3. تحميلُ إشارةِ EEG

نحمّلُ تسجيلَ المُشاركِ 1 في التجربةِ 1، الجلسةِ 2. تَحتوي الإشارةُ على 4 قنواتٍ (P4, Cz, F8, T7) مُسجَّلةٍ بمعدّلِ 200 Hz. سنعملُ على قناةِ P4 (المنطقةُ الجداريةُ) في هذا المثالِ.

</div>

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=7, experiment=1, session=2
)
channel_data = eeg_data[:, 0]  # P4 channel
fs = 200  # Sampling rate (Hz)

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')
print(f'Sampling rate: {fs} Hz')

<div dir="rtl" align="right">

## 4. تطبيقُ مُرشِّحِ التردداتِ العاليةِ

نَستخدمُ مُرشِّحَ باتروورث من الترتيبِ الرابعِ بترددِ قطعٍ 1 Hz. تُطبِّقُ دالةُ `filtfilt` المُرشِّحَ مرّتينِ (أمامًا ثمَّ خلفًا)، فتُلغي أيَّ تأخيرٍ في الطورِ. هذا ضروريٌّ في تحليلِ EEG حيثُ يُهمُّ التوقيتُ الدقيقُ.

</div>

In [ ]:
from scipy import signal

def butter_highpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs  # Nyquist frequency = 100 Hz
    normal_cutoff = cutoff / nyq
    b, a = signal.butter(order, normal_cutoff, btype='high', analog=False)
    filtered = signal.filtfilt(b, a, data)
    return filtered

filtered_hp = butter_highpass_filter(channel_data, cutoff=1.0, fs=fs)
print(f'Filter applied: high-pass at {1.0} Hz, order {4}')
print(f'Nyquist frequency: {0.5*fs} Hz')

<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ: قبلَ الترشيحِ وبعدَهُ

الرسمُ أدناهُ تفاعليٌّ. يمكنكَ:
- التكبيرُ بالنقرِ والسحبِ
- التنقّلُ بأداةِ Pan
- تمريرُ المؤشّرِ فوقَ النقاطِ لرؤيةِ القيمِ الدقيقةِ

**علامَ تُلاحظُ؟**
- الإشارةُ الخامُ (أعلى) فيها انجرافٌ بطيءٌ يظهرُ كميلٍ تدريجيٍّ نحوَ الأعلى أو الأسفلِ
- الإشارةُ المُرشَّحةُ (أسفل) مُتمركزةٌ حولَ الصفرِ، واختفى الانجرافُ
- الإشارةُ المُرشَّحةُ لا تزالُ مُشوّشةً (تذبذباتٌ عاليةُ الترددِ) وهذا متوقَّعٌ، سيزيلُها مُرشِّحُ التردداتِ المنخفضةِ

</div>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

n_plot = min(5000, len(channel_data))
t_sec = timestamps[:n_plot] / 1000.0  # convert ms to seconds

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Raw EEG (P4)', 'After high-pass filter (1 Hz)'))

fig.add_trace(go.Scatter(x=t_sec, y=channel_data[:n_plot],
                         name='Raw', line=dict(color='gray', width=0.5)),
               row=1, col=1)
fig.add_trace(go.Scatter(x=t_sec, y=filtered_hp[:n_plot],
                         name='Filtered', line=dict(color='green', width=0.5)),
               row=2, col=1)

fig.update_layout(height=600, title_text='High-Pass Filter: Before vs After',
                  xaxis2_title='Time (s)', yaxis_title='EEG (uV)',
                  yaxis2_title='EEG (uV)')
fig.show()

<div dir="rtl" align="right">

## 6. خلاصةٌ

- أزالَ مُرشِّحُ التردداتِ العاليةِ عندَ 1 Hz إزاحةَ التيارِ المُستمرِّ والانجرافَ البطيءَ من الإشارةِ
- صارتِ الإشارةُ مُتمركزةً حولَ الصفرِ على المحورِ العموديِّ
- الضجيجُ عاليُّ الترددِ لا يزالُ موجودًا، وتبدو الإشارةُ متعرّجةً
- الخطوةُ التاليةُ هي تطبيقُ مُرشِّحِ التردداتِ المنخفضةِ لإزالةِ ذلكَ الضجيجِ (انظرِ الدفترَ التاليَ)

</div>